In [1]:
import argparse
import json
import re
import os
import torch
import traceback
import numpy as np
from datetime import datetime
from typing import Dict, List, Any
from pathlib import Path
from tqdm import tqdm
from argparse import Namespace

# Set correct directory pathing
import os
import sys

# Import project modules
sys.path.insert(0, '/Users/wpang/Desktop/rare_disease_pyHealth/RDMA')
from rdma.rdrag.entity import LLMRDExtractor, RetrievalEnhancedRDExtractor,MultiIterativeRDExtractor,IterativeLLMRDExtractor
from rdma.utils.embedding import EmbeddingsManager
from rdma.hporag.context import ContextExtractor
from rdma.utils.llm_client import LocalLLMClient, APILLMClient
from rdma.utils.setup import setup_device
from dotenv import load_dotenv

load_dotenv()

/Users/wpang/Desktop/rare_disease_pyHealth/.venv/lib/python3.12/site-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


True

In [2]:
import pandas as pd

pd.set_option('display.max_colwidth', None)

In [3]:
SAMPLE_SIZE = 5

### GettingStarted

In [4]:
from pyhealth.datasets.mimic4 import MIMIC4NoteDataset

In [5]:
NOTE_ROOT = '/Users/wpang/Desktop/rare_disease_pyHealth/RDMA/notebooks/wp'

dataset = MIMIC4NoteDataset(root=NOTE_ROOT, tables=["discharge"])
note_df = dataset.global_event_df.collect().to_pandas()

Using default note config: /Users/wpang/Desktop/rare_disease_pyHealth/.venv/lib/python3.12/site-packages/pyhealth/datasets/configs/mimic4_note.yaml
Memory usage Before initializing mimic4_note: 572.9 MB
Initializing mimic4_note dataset from /Users/wpang/Desktop/rare_disease_pyHealth/RDMA/notebooks/wp (dev mode: False)
Memory usage After initializing mimic4_note: 573.1 MB
No cache_dir provided. Using default cache dir: /Users/wpang/Library/Caches/pyhealth/39ebf87e-69d9-56a7-bd4d-af1779f39061


/Users/wpang/Desktop/rare_disease_pyHealth/.venv/lib/python3.12/site-packages/pyhealth/datasets/mimic4.py:103: UserWarning: Events from discharge table only have date timestamp (no specific time). This may affect temporal ordering of events.
  warnings.warn(


In [6]:
patient_notes = (
    note_df
    .sort_values("timestamp")
    .groupby("patient_id")
    .apply(
        lambda x: dict(zip(x["timestamp"], x["discharge/text"]))
     )
    )

samples = patient_notes.sample(n=SAMPLE_SIZE, random_state=42)

/var/folders/cr/h1qm2rws041_nzss3n7df3vm0000gn/T/ipykernel_11640/4148831802.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


In [7]:
final_notes = {
    str(patient_id): {
        str(charttime): {"note_content": text} 
        for charttime, text in notes.items()
    }
    for patient_id, notes in samples.items()
}

## Extract Rare Disease

In [8]:
args = Namespace(
      llm_type="api",
      api_config="api_config.json"
  )

def initialize_llm_client(args: argparse.Namespace):
    """Initialize appropriate LLM client based on arguments."""
    if args.llm_type == "api":
        if args.api_config:
            return APILLMClient.from_config(args.api_config)
        else:
            return APILLMClient.initialize_from_input()
    else:  # local
        return LocalLLMClient(
            model_type=args.model_type,
            device=device,
            cache_dir=args.cache_dir,
            temperature=args.temperature
        )

llm_client = initialize_llm_client(args)        

entity_extractor = LLMRDExtractor(
      llm_client=llm_client,
      system_message="You are a medical expert specializing in rare diseases."
  )

## Getting context from entities

In [9]:
class CustomContextExtractor:
    def extract_sentences(self, text: str) -> List[str]:
        
        # First split by common sentence terminators while preserving them
        sentence_parts = []
        for part in re.split(r"([.!?])", text):
            if part.strip():
                if part in ".!?":
                    if sentence_parts:
                        sentence_parts[-1] += part
                else:
                    sentence_parts.append(part.strip())

        # Then handle other clinical note delimiters like line breaks and semicolons
        sentences = []
        for part in sentence_parts:
            # Split by semicolons and newlines
            for subpart in re.split(r"[;\n]", part):
                if subpart.strip():
                    sentences.append(subpart.strip())

        return sentences

    def find_entity_context(self, entity: str, sentences: List[str], window_size):
        entity_lower = entity.lower()
        for i, sentence in enumerate(sentences):
            if entity_lower in sentence.lower():
                # Found exact match - include surrounding sentences based on window_size
                return self.get_context_window(sentences, i, window_size)

        # More sophisitication: Use some sort of fuzzy matching. 
        # This requires iterating over each sentence, and then checking
        # if each word in the sentence "fuzzy" matches the entity. If it fuzzy matches above
        # a threshold, then we have found a match and we keep the index (where the sentence came
        # from)

        # Psuedocode
        # entity_words = set(re.findall(r"\b\w+\b", entity_lower))
        # for i, sentence in enumerate(sentences):
        #     sentence_words = set(re.findall(r"\b\w+\b"), entity_lower)

        #     common_words = entity_words & sentence_words

        #     best_match_i = i # This updates as a better match gets found

        #  return self.get_context_window(sentences, best_match_i, window_size)

    def get_context_window(self, sentences: List[str], center_index: int, window_size: int):
        start_index = max(0, center_index - window_size)
        end_index = min(len(sentences) - 1, center_index + window_size)
        context_sentences = sentences[start_index : end_index + 1]

        return " ".join(context_sentences).strip()

    def extract_context(self, entities: List[str], text: str, window_size: int = 0):

        sentences = self.extract_sentences(text)

        results = []
        for entity in entities:
            context = self.find_entity_context(entity, sentences, window_size)
            results.append(
                {
                    "entity": entity,
                    "context": context or "",  # Empty string if no context found
                }
            )

        return results

In [10]:
context_extractor = CustomContextExtractor()

for i, (patient_id, patient_data) in enumerate(tqdm(list(final_notes.items()), desc="Processing cases")):
    for charttime, note in patient_data.items():
        note['llm_extracted_entities'] = entity_extractor.extract_entities(note['note_content'])
        note['entity_context'] = context_extractor.extract_context(note['llm_extracted_entities'], note['note_content'], window_size=0)

Processing cases:   0%|                                                                         | 0/5 [00:00<?, ?it/s]

TOTAL_TOKENS_USED before query: 0


Processing cases:  20%|█████████████                                                    | 1/5 [00:01<00:05,  1.35s/it]

TOTAL_TOKENS_USED before query: 3954


Processing cases:  40%|██████████████████████████                                       | 2/5 [00:01<00:02,  1.13it/s]

...

Processing cases:  60%|███████████████████████████████████████                          | 3/5 [00:02<00:01,  1.27it/s]

TOTAL_TOKENS_USED before query: 12020


Processing cases:  80%|████████████████████████████████████████████████████             | 4/5 [00:03<00:00,  1.24it/s]

TOTAL_TOKENS_USED before query: 16045
TOTAL_TOKENS_USED before query: 18714


Processing cases: 100%|█████████████████████████████████████████████████████████████████| 5/5 [00:04<00:00,  1.19it/s]


In [15]:
test_case = final_notes['13106750']['2117-04-16 00:00:00']['entity_context']
print(test_case)

[{'entity': 'Squamos cell carcinoma', 'context': 'Squamos cell carcinoma of epiglottis, treated with'}, {'entity': 'Adenocarcinoma', 'context': 'completed ___ ___, adenocarcinoma of stage IV left lung'}, {'entity': 'COPD', 'context': 'COPD'}, {'entity': 'CKD', 'context': 'CKD stage III (baseline 1.'}, {'entity': 'Hypertension', 'context': 'Hypertension'}, {'entity': 'Hyperlipidemia', 'context': 'Hyperlipidemia'}, {'entity': 'GERD', 'context': 'GERD'}, {'entity': 'Hematuria', 'context': 'History of microscopic hematuria with known uric acid'}, {'entity': 'Uric acid nephrolithiasis', 'context': ''}, {'entity': 'Thrombocytopenia', 'context': 'Course complicated by thrombocytopenia and was diagnosed with'}, {'entity': 'HIT', 'context': 'HIT with positive antibody assay.'}, {'entity': 'Sepsis', 'context': 'sepsis.'}, {'entity': 'Pneumonia', 'context': 'found to have sepesis secondary to bilateral pneumonia and'}, {'entity': 'Acute on chronic renal failure', 'context': 'then developed acute 